# Global LSTM-Autoencoder for cross-PLC anomaly detection.
Trains on `normal.csv` only; evaluates on `attack.csv`.

In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping



## STEP 1 — LOAD DATA

In [ ]:
normal_df = pd.read_csv("normal.csv")
attack_df = pd.read_csv("attack.csv")

# Strip whitespace from column names (common SWaT gotcha)
normal_df.columns = normal_df.columns.str.strip()
attack_df.columns = attack_df.columns.str.strip()

# Parse timestamp if present, sort chronologically (do NOT shuffle raw rows)
TIMESTAMP_COL = "Timestamp"   # adjust to your actual column name
if TIMESTAMP_COL in normal_df.columns:
    normal_df[TIMESTAMP_COL] = pd.to_datetime(normal_df[TIMESTAMP_COL].str.strip(), format='mixed', dayfirst=True)
    normal_df = normal_df.sort_values(TIMESTAMP_COL).reset_index(drop=True)
if TIMESTAMP_COL in attack_df.columns:
    attack_df[TIMESTAMP_COL] = pd.to_datetime(attack_df[TIMESTAMP_COL].str.strip(), format='mixed', dayfirst=True)
    attack_df = attack_df.sort_values(TIMESTAMP_COL).reset_index(drop=True)


## STEP 2 — LABEL HANDLING (attack.csv only)

In [ ]:
LABEL_COL = "attack"   # adjust to your actual column name
attack_df[LABEL_COL] = attack_df[LABEL_COL].astype(str).str.strip()
attack_df["label"] = attack_df[LABEL_COL].map(
    lambda x: 0 if str(x).strip().lower() in ["normal", "0", "0.0", "false"] else 1
)


## STEP 3 — IDENTIFY FEATURE COLUMNS (drop timestamp/label)

In [ ]:
exclude_cols = [TIMESTAMP_COL, LABEL_COL, "label"]
feature_cols = [c for c in normal_df.columns if c not in exclude_cols]

print(f"Number of feature columns (all PLCs combined): {len(feature_cols)}")

# Remove any zero-variance (static) columns, computed on normal_df ONLY
train_std = normal_df[feature_cols].std()
static_cols = train_std[train_std < 1e-6].index.tolist()
if static_cols:
    print(f"Dropping {len(static_cols)} static columns: {static_cols}")
    feature_cols = [c for c in feature_cols if c not in static_cols]


## STEP 4 — TRAIN / VAL / TEST SPLIT

In [ ]:
# normal.csv -> train (85%) + val (15%), chronological, no shuffle
split_idx = int(len(normal_df) * 0.85)
train_df = normal_df.iloc[:split_idx].reset_index(drop=True)
val_df   = normal_df.iloc[split_idx:].reset_index(drop=True)

# attack.csv -> test (untouched, used only for final evaluation)
test_df = attack_df.reset_index(drop=True)

print(f"Train rows: {len(train_df)} | Val rows: {len(val_df)} | Test rows: {len(test_df)}")


## STEP 5 — SCALING (fit on train ONLY)

In [ ]:
scaler = MinMaxScaler()
scaler.fit(train_df[feature_cols])

train_scaled = scaler.transform(train_df[feature_cols])
val_scaled   = scaler.transform(val_df[feature_cols])
test_scaled  = scaler.transform(test_df[feature_cols])

joblib.dump(scaler, "global_lstm_scaler.pkl")
joblib.dump(feature_cols, "global_lstm_feature_cols.pkl")


## STEP 6 — WINDOWING

In [ ]:
WINDOW_SIZE = 60   # tune: ~1 min of data at 1Hz sampling; adjust to your rate
STRIDE = 1
BATCH_SIZE = 64

def make_dataset(data, window_size, stride, batch_size, shuffle=False):
    ds = tf.keras.utils.timeseries_dataset_from_array(
        data=data,
        targets=None,
        sequence_length=window_size,
        sequence_stride=stride,
        batch_size=batch_size,
        shuffle=shuffle
    )
    # Autoencoder targets itself
    ds = ds.map(lambda x: (x, x), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_dataset = make_dataset(train_scaled, WINDOW_SIZE, STRIDE, BATCH_SIZE, shuffle=True)
val_dataset = make_dataset(val_scaled, WINDOW_SIZE, STRIDE, BATCH_SIZE)
test_dataset = make_dataset(test_scaled, WINDOW_SIZE, STRIDE, BATCH_SIZE)

num_features = train_scaled.shape[1]


## STEP 7 — MODEL ARCHITECTURE

In [ ]:
def build_lstm_autoencoder(window_size, num_features, latent_dim=32):
    inputs = layers.Input(shape=(window_size, num_features))
    encoded = layers.LSTM(latent_dim, activation="tanh", return_sequences=False)(inputs)
    repeated = layers.RepeatVector(window_size)(encoded)
    decoded = layers.LSTM(latent_dim, activation="tanh", return_sequences=True)(repeated)
    outputs = layers.TimeDistributed(layers.Dense(num_features))(decoded)
    model = Model(inputs, outputs, name="global_lstm_autoencoder")
    model.compile(optimizer="adam", loss="mse")
    return model

autoencoder = build_lstm_autoencoder(WINDOW_SIZE, num_features, latent_dim=32)
autoencoder.summary()


## STEP 8 — TRAIN (normal data only)

In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

history = autoencoder.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    callbacks=[early_stop],
    verbose=2
)

autoencoder.save("global_lstm_autoencoder.keras")


## STEP 9 — THRESHOLD CALIBRATION (on val_df, still normal-only)

In [ ]:
def calculate_mse_from_dataset(model, dataset):
    mse_list = []
    for x_batch, _ in dataset:
        recon_batch = model(x_batch, training=False)
        mse = tf.reduce_mean(tf.square(tf.cast(x_batch, tf.float32) - recon_batch), axis=[1, 2])
        mse_list.append(mse.numpy())
    return np.concatenate(mse_list)

val_mse = calculate_mse_from_dataset(autoencoder, val_dataset)   # per-window MSE

THRESHOLD_PERCENTILE = 99
threshold = np.percentile(val_mse, THRESHOLD_PERCENTILE)

print(f"Val MSE stats -> mean: {val_mse.mean():.6f}, std: {val_mse.std():.6f}")
print(f"Calibrated threshold (P{THRESHOLD_PERCENTILE}): {threshold:.6f}")

joblib.dump({"threshold": threshold, "window_size": WINDOW_SIZE, "stride": STRIDE},
            "global_lstm_threshold.pkl")


## STEP 10 — TEST (attack.csv)

In [ ]:
test_mse = calculate_mse_from_dataset(autoencoder, test_dataset)
test_flags = test_mse > threshold

# ------------------------------------------------------------
# Map window-level flags back to per-timestep labels.
# Convention: each window's flag is assigned to its LAST timestep index.
# ------------------------------------------------------------
n_windows = len(test_flags)
window_end_indices = np.array([i * STRIDE + WINDOW_SIZE - 1 for i in range(n_windows)])

timestep_flags = np.zeros(len(test_df), dtype=bool)
timestep_scores = np.full(len(test_df), np.nan)
timestep_flags[window_end_indices] = test_flags
timestep_scores[window_end_indices] = test_mse

# forward-fill scores/flags so every timestep between window-ends has a value
# (alternative: leave as NaN and drop those rows before scoring — document your choice)
results_df = pd.DataFrame({
    "timestamp": test_df[TIMESTAMP_COL] if TIMESTAMP_COL in test_df.columns else np.arange(len(test_df)),
    "ground_truth_label": test_df["label"].values,
    "lstm_flag": timestep_flags,
    "lstm_score": timestep_scores
})
results_df["lstm_score"] = results_df["lstm_score"].ffill()
results_df["lstm_flag"] = results_df["lstm_flag"].astype(bool)

results_df.to_csv("global_lstm_results.csv", index=False)


## STEP 11 — EVALUATION METRICS

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, confusion_matrix

# only score rows where we actually have a window-derived value
valid_mask = results_df["lstm_score"].notna()
y_true = results_df.loc[valid_mask, "ground_truth_label"].values
y_pred = results_df.loc[valid_mask, "lstm_flag"].astype(int).values
y_score = results_df.loc[valid_mask, "lstm_score"].values

precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
auprc = average_precision_score(y_true, y_score)

fp = np.sum((y_pred == 1) & (y_true == 0))
tn = np.sum((y_pred == 0) & (y_true == 0))
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

print("\n===== Global LSTM-Autoencoder Results =====")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"FPR:       {fpr:.4f}")
print(f"AUPRC:     {auprc:.4f}")

cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:")
print(cm)

metrics_summary = {
    "precision": precision, "recall": recall, "f1": f1,
    "fpr": fpr, "auprc": auprc,
    "threshold": float(threshold), "window_size": WINDOW_SIZE
}
joblib.dump(metrics_summary, "global_lstm_metrics.pkl")

In [ ]:
# --- Test ONLY on attack.csv ---
print("Evaluating LSTM AE ONLY on attack.csv...")
# As shown above, the test_dataset is already generated strictly from attack.csv
# So we can just reprint the metrics using valid_mask on results_df

from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

valid_mask = results_df["lstm_score"].notna()
y_true_atk = results_df.loc[valid_mask, "ground_truth_label"].values
y_pred_atk = results_df.loc[valid_mask, "lstm_flag"].astype(int).values

print("Classification Report on attack.csv:")
print(classification_report(y_true_atk, y_pred_atk))
print("Confusion Matrix on attack.csv:")
print(confusion_matrix(y_true_atk, y_pred_atk))
